# Consumer Discretionary: FinBERT scoring + Model C

Runtime > Change runtime type > **GPU** (T4 takes about 2 h at the IT run's ~190 sentences/s; L4/A100 is faster).

1. Put `cd_finbert_input.zip` anywhere in My Drive, zipped or already unzipped. It holds licensed WRDS transcripts: keep it private and delete it afterwards.
2. Run all cells. Finished years are checkpointed to Drive, so after a disconnect just Run all again.
3. Download `MyDrive/cd_finbert/cd_finbert_output.zip` and unzip it into `cd_results/`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import torch
assert torch.cuda.is_available(), "Select a GPU runtime first"
print(torch.cuda.get_device_name(0))

In [ ]:
!pip -q install transformers xgboost pyarrow scikit-learn
# Find the inputs anywhere in My Drive (zip, or an already-unzipped folder) and copy them to /content/cd
import glob, os, shutil, zipfile
W = "/content/cd"
shutil.rmtree(W, ignore_errors=True)
os.makedirs("/content/drive/MyDrive/cd_finbert", exist_ok=True)
zips = glob.glob("/content/drive/MyDrive/**/cd_finbert_input*.zip", recursive=True)
scripts = [s for s in glob.glob("/content/drive/MyDrive/**/it_finbert_features.py", recursive=True)
           if os.path.isdir(os.path.join(os.path.dirname(s), "earnings_sp500", "text"))]
if zips:
    print("using zip:", zips[0])
    zipfile.ZipFile(zips[0]).extractall(W)
elif scripts:
    print("using unzipped folder:", os.path.dirname(scripts[0]))
    shutil.copytree(os.path.dirname(scripts[0]), W)
else:
    raise FileNotFoundError("cd_finbert_input.zip not found in My Drive (it can take a minute to appear after upload)")
# a zip re-made by hand may add one extra top-level folder
if not os.path.exists(f"{W}/it_finbert_features.py"):
    inner = [d for d in glob.glob(f"{W}/*/") if os.path.exists(d + "it_finbert_features.py")]
    assert inner, f"it_finbert_features.py not in the input; found: {os.listdir(W)}"
    for x in os.listdir(inner[0]):
        shutil.move(inner[0] + x, W)
for need in ["it_finbert_features.py", "it_model_c_finbert.py", "earnings_sp500/event_panel.parquet",
             "earnings_cd/model_table_opt1.parquet", "earnings_cd/model_table_opt2.parquet"]:
    assert os.path.exists(f"{W}/{need}"), f"missing {need}"
n = len(glob.glob(f"{W}/earnings_sp500/text/text_*.parquet"))
assert n == 17, f"expected 17 yearly text files, found {n}"
print("inputs OK:", sorted(os.listdir(W)))

In [ ]:
# FinBERT features (same script and settings as the IT run)
!cd /content/cd && python it_finbert_features.py --sp500-folder earnings_sp500 --it-folder earnings_cd --batch-size 128 --checkpoint-dir /content/drive/MyDrive/cd_finbert/ckpt 2>&1 | tee -a earnings_cd/cd_finbert_run.log
import os
assert os.path.exists("/content/cd/earnings_cd/finbert_features.parquet"), "FinBERT step failed: read the log above"

In [ ]:
# Check before modelling
import pandas as pd
f = pd.read_parquet("/content/cd/earnings_cd/finbert_features.parquet")
print(f.shape, f.transcriptid.is_unique)
assert len(f) == 3710 and f.transcriptid.is_unique
print(f.describe().T)

In [ ]:
# Model C (includes XGBoost, as in the IT run)
!cd /content/cd && python it_model_c_finbert.py --folder earnings_cd --prefix cd --sector-label CD 2>&1 | tee earnings_cd/cd_modelc_run.log
import glob
assert glob.glob("/content/cd/earnings_cd/cd_model_c_*_report.txt"), "Model C failed: read the log above"

In [ ]:
import pandas as pd
f = pd.read_parquet("/content/cd/earnings_cd/finbert_features.parquet")
f.to_csv("/content/cd/earnings_cd/finbert_features.csv", index=False)
!cd /content/cd/earnings_cd && zip -q /content/drive/MyDrive/cd_finbert/cd_finbert_output.zip finbert_features.* cd_model_c_* cd_finbert_run.log cd_modelc_run.log
!unzip -l /content/drive/MyDrive/cd_finbert/cd_finbert_output.zip